Analyze Suricata Rules

In [ ]:
%pip install pandas matplotlib

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------------------------------------------------------
# 1. Setup and Data Loading
# -----------------------------------------------------------------------------
sys.path.append(str(Path("parse_suricata_rules").resolve().parent.parent / "training-data/scripts"))

try:
    import parse_suricata_rules as psr
    print("Loading DataFrame...")
    df = psr.parse_to_dataframe('../training-data/sample_rules/valid_online.rules')
    # df = psr.parse_to_dataframe('../training-data/dataset_rule_check/good.rules')
except ImportError:
    print("Error: parse_suricata_rules not found. Please verify the path.")
    sys.exit(1)

print(f"Total rules loaded: {len(df)}")

# Helper function to map dictionary keywords to the DataFrame's column format
def get_valid_cols(keywords):
    valid_cols = []
    for kw in keywords:
        # Normalize keyword to match DataFrame columns: 'http.uri' -> 'opt_http_uri'
        col_name = f"opt_{kw.replace('.', '_').replace('-', '_')}"
        if col_name in df.columns:
            valid_cols.append(col_name)
    return valid_cols

# -----------------------------------------------------------------------------
# 2. Definitions
# -----------------------------------------------------------------------------
CATEGORY_KEYWORDS = {
    'Protocol-Based Logic': ['app-layer-protocol', 'http_stat_code', 'http_method', 'dns_query', 'tls_sni'],
    'Signature-Based (Pattern)': ['content', 'pcre', 'byte_test', 'byte_jump', 'byte_extract'],
    'Flow & State-Based Logic': ['flow', 'flowbits', 'stream_size'],
    'Metadata & File-Based Logic': ['filename', 'fileext', 'filemagic', 'filemd5', 'file_data', 'filestore'],
    'Anomaly-Based Logic': ['app-layer-event', 'threshold', 'limit', 'dsize', 'detection_filter']
}

KEYWORD_CATEGORIES = {
    'http1': {"http_uri", "http_raw_uri", "http_method", "http_header", "http_raw_header", "http_cookie", "http_client_body", "http_stat_code", "http_stat_msg", "http_user_agent", "http_host", "http_raw_host", "uricontent", "urilen"},
    'http2': {"http.uri", "http.uri.raw", "http.host", "http.host.raw", "http.method", "http.request_line", "http.request_body", "http.response_line", "http.response_body", "http.header", "http.header.raw", "http.header_names", "http.cookie", "http.user_agent", "http.accept", "http.accept_enc", "http.accept_lang", "http.referer", "http.connection", "http.content_len", "http.content_type", "http.location", "http.server", "http.protocol", "http.stat_code", "http.stat_msg", "http.start", "http.request_header", "http.response_header", "file.data", "file.name"},
    'dns': {"dns.query", "dns.answer", "dns.answer.name", "dns.authority", "dns.authority.name", "dns.opcode", "dns.rrtype"},
    'tls': {"tls.cert_subject", "tls.cert_issuer", "tls.cert_serial", "tls.cert_fingerprint", "tls.sni", "tls.certs", "tls.version", "tls.subject", "tls.issuerdn", "tls.cert_chain_len", "tls.cert_notbefore", "tls.cert_notafter", "tls.random", "tls.alpn", "ja3.hash", "ja3.string", "ja3s.hash", "ja3s.string"},
    'tls_legacy': {"ssl_version", "ssl_state", "tls_sni", "tls_cert_subject", "tls_cert_issuer", "tls_cert_serial", "tls_cert_fingerprint"},
    'ssh': {"ssh.proto", "ssh.software", "ssh.hassh", "ssh.hassh.string", "ssh.hassh.server", "ssh.hassh.server.string", "ssh_proto", "ssh_software"},
    'smtp': {"smtp.helo", "smtp.mail_from", "smtp.rcpt_to", "app-layer-event:smtp"},
    'packet': {"dsize", "flags", "ttl", "id", "ipopts", "fragbits", "fragoffset", "tos", "seq", "ack", "window"},
}

TARGET_PROTOCOLS = [
    "ip", "tcp", "udp", "icmp", "tcp-pkt", "tcp-stream", "pkthdr", "http", "http1", "http2", "ftp", "ftp-data", "tls", "ssl", "smb", "dns", "dcerpc", "dhcp", "ssh", "smtp", "imap", "pop3", "modbus", "dnp3", "enip", "nfs", "ike", "krb5", "bittorrent-dht", "ntp", "rfb", "rdp", "snmp", "tftp", "sip", "websocket", "quic", "mqtt", "http_any", "pgsql", "ja4", "mdns"
]

# -----------------------------------------------------------------------------
# 3. Data Processing
# -----------------------------------------------------------------------------

# --- 1) Logic Categories ---
logic_counts = {}
for cat, kws in CATEGORY_KEYWORDS.items():
    valid_cols = get_valid_cols(kws)
    if valid_cols:
        logic_counts[cat] = df[valid_cols].notna().any(axis=1).sum()
    else:
        logic_counts[cat] = 0

# --- 2) Action Types ---
if 'action' in df.columns:
    action_counts = df['action'].str.lower().value_counts()
else:
    action_counts = pd.Series(dtype=int)

# --- 3) Protocols (> 0.5%) ---
if 'protocol' in df.columns:
    # Normalize protocols and filter against TARGET_PROTOCOLS
    protocols_cleaned = df['protocol'].str.lower()
    proto_freq = protocols_cleaned.value_counts(normalize=True) * 100
    # Keep only those strictly > 0.5% and existing in the target list
    top_protos = proto_freq[(proto_freq > 0.5) & (proto_freq.index.isin(TARGET_PROTOCOLS))]
    
    # Calculate "Other" for protocols that didn't make the 0.5% cut
    other_pct = 100 - top_protos.sum()
    if other_pct > 0:
        top_protos['other (< 0.5%)'] = other_pct
else:
    top_protos = pd.Series(dtype=float)

# --- 4) Suricata Keyword Categories ---
keyword_counts = {}
for cat, kws in KEYWORD_CATEGORIES.items():
    valid_cols = get_valid_cols(kws)
    if valid_cols:
        # If any of the mapped columns have a non-null value, it counts for this category
        keyword_counts[cat] = df[valid_cols].notna().any(axis=1).sum()
    else:
        keyword_counts[cat] = 0

# --- 5) Syntax Comparison ---
# Helper logic to parse contents based on your sample dataframe formatting
def is_multiple_content(val):
    if pd.isna(val): return False
    val_str = str(val)
    # The parser uses pipe/quotes (e.g. """|b5 76|""|""User-Agent""") to delimit multiple contents
    return '""|""' in val_str or '","' in val_str

has_prefilter = df['opt_prefilter'].notna() & (df['opt_prefilter'] != '') if 'opt_prefilter' in df.columns else pd.Series(False, index=df.index)
has_fast_pattern = df['opt_fast_pattern'].notna() & (df['opt_fast_pattern'] != '') if 'opt_fast_pattern' in df.columns else pd.Series(False, index=df.index)
has_pcre = df['opt_pcre'].notna() & (df['opt_pcre'] != '') if 'opt_pcre' in df.columns else pd.Series(False, index=df.index)
has_content = df['opt_content'].notna() & (df['opt_content'] != '') if 'opt_content' in df.columns else pd.Series(False, index=df.index)

multiple_content = df.get('opt_content', pd.Series([False]*len(df))).apply(is_multiple_content)
single_content = has_content & ~multiple_content
no_content = ~has_content
pcre_and_content = has_pcre & has_content

syntax_counts = {
    'Pre-Filter': has_prefilter.sum(),
    'Fast Pattern': has_fast_pattern.sum(),
    'Single Content': single_content.sum(),
    'Multiple Content': multiple_content.sum(),
    'No Content': no_content.sum(),
    'PCRE': has_pcre.sum(),
    'PCRE & Content': pcre_and_content.sum()
}

# -----------------------------------------------------------------------------
# 4. Visualization Generation
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(18, 16))
plt.style.use('ggplot')

# Chart 1: Logic Categories
ax1 = plt.subplot(3, 2, 1)
pd.Series(logic_counts).sort_values().plot(kind='barh', ax=ax1, color='#3498db', edgecolor='black')
ax1.set_title('1. Suricata Rules by General Logic', fontweight='bold')
ax1.set_xlabel('Rule Count')

# Chart 2: Action Types
ax2 = plt.subplot(3, 2, 2)
if not action_counts.empty:
    action_counts.plot(kind='bar', ax=ax2, color='#e74c3c', edgecolor='black')
    ax2.set_title('2. Rules by Action Type', fontweight='bold')
    ax2.set_ylabel('Rule Count')
    ax2.tick_params(axis='x', rotation=0)

# Chart 3: Protocols Pie Chart
ax3 = plt.subplot(3, 2, 3)
if not top_protos.empty:
    top_protos.plot(kind='pie', ax=ax3, autopct='%1.1f%%', startangle=140, 
                    colors=plt.cm.Paired(np.linspace(0, 1, len(top_protos))),
                    wedgeprops={'edgecolor': 'black'})
    ax3.set_title('3. Target Protocols (> 0.5% occurrence)', fontweight='bold')
    ax3.set_ylabel('')

# Chart 4: Keyword Categories
ax4 = plt.subplot(3, 2, 4)
pd.Series(keyword_counts).sort_values(ascending=False).plot(kind='bar', ax=ax4, color='#2ecc71', edgecolor='black')
ax4.set_title('4. Keyword Match Categories', fontweight='bold')
ax4.set_ylabel('Rule Count')
ax4.tick_params(axis='x', rotation=45)

# Chart 5: Syntax Comparison
ax5 = plt.subplot(3, 1, 3) # Spans the bottom row
pd.Series(syntax_counts).plot(kind='bar', ax=ax5, color='#9b59b6', edgecolor='black')
ax5.set_title('5. Rule Syntax & Payload Inspection Attributes', fontweight='bold')
ax5.set_ylabel('Rule Count')
ax5.tick_params(axis='x', rotation=0)

# Add value labels on top of the bars for the Syntax chart
for container in ax5.containers:
    ax5.bar_label(container, padding=3)

plt.tight_layout()
plt.show()

print("\nAnalysis complete! Visualizations saved to 'suricata_dataframe_analysis.png'.")

Suricata Rules Pattern Fingerprint Analysis

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

def analyze_suricata_patterns(df):
    """
    Analyzed pre-parsed Suricata rules from DataFrame columns to determine 
    syntax patterns, complexity, and structural abnormalities.
    """
    
    # Identify which columns represent Suricata options (those starting with opt_)
    opt_cols = [c for c in df.columns if c.startswith('opt_')]
    
    def get_row_stats(row):
        # 1. Determine active keywords (where the column is not null/empty)
        # We strip the 'opt_' prefix to keep the sequence clean
        active_keywords = [col.replace('opt_', '') for col in opt_cols 
                           if pd.notnull(row[col]) and str(row[col]).strip() != '']
        
        # 2. Estimate Rule Length (Header + Options string)
        # This helps keep the "is_compressed" and "rule_length" stats working
        header_str = f"{row['action']} {row['protocol']} {row['src_address']} {row['src_port']} {row['direction']} {row['dst_address']} {row['dst_port']}"
        options_str = "; ".join([f"{k}:{row['opt_'+k]}" for k in active_keywords])
        full_rule_estimate = f"{header_str} ({options_str})"
        
        # 3. Extract SID safely
        try:
            sid = int(float(row['opt_sid'])) if pd.notnull(row['opt_sid']) else None
        except:
            sid = None

        return {
            "proto": row['protocol'],
            "direction": row['direction'],
            "keyword_sequence": "-".join(active_keywords),
            "keyword_count": len(active_keywords),
            "rule_length": len(full_rule_estimate),
            "sid": sid,
            "has_pcre": pd.notnull(row.get('opt_pcre')) and str(row.get('opt_pcre')).strip() != '',
            "is_compressed": len(options_str) < 450 and pd.notnull(row.get('opt_msg'))
        }

    print("Analyzing rule structures from DataFrame columns...")
    
    # Apply logic to each row
    analysis_results = df.apply(get_row_stats, axis=1)
    analysis_df = pd.DataFrame(list(analysis_results))

    # --- Reports (Unaltered from your original requirement) ---

    # 1. Top Syntax Patterns
    print("\n--- TOP SYNTAX PATTERNS (Keyword Presence) ---")
    patterns = analysis_df['keyword_sequence'].value_counts().head(10)
    print(patterns)

    # 2. SID Distribution Analysis
    if 'sid' in analysis_df.columns and not analysis_df['sid'].isnull().all():
        print("\n--- SID RANGE ANALYSIS ---")
        print(f"Min SID: {analysis_df['sid'].min()}")
        print(f"Max SID: {analysis_df['sid'].max()}")
        
        valid_sids = analysis_df.dropna(subset=['sid'])
        seven_digit_sids = valid_sids[valid_sids['sid'].between(1000000, 9999999)].shape[0]
        print(f"Rules with 7-digit SIDs: {seven_digit_sids} ({seven_digit_sids/len(analysis_df)*100:.2f}%)")

    # 3. Protocol Variation
    print("\n--- PROTOCOL DISTRIBUTION ---")
    print(analysis_df['proto'].value_counts())

    # 4. Complexity Stats for Qwen/CodeBERT
    print("\n--- COMPLEXITY METRICS ---")
    print(f"Average Keyword Count: {analysis_df['keyword_count'].mean():.2f}")
    print(f"Rules with PCRE: {analysis_df['has_pcre'].sum()}")
    
    return analysis_df

# --- Usage ---
sys.path.append(str(Path("parse_suricata_rules").resolve().parent.parent / "training-data/scripts"))

try:
    import parse_suricata_rules as psr
    print("Loading DataFrame...")
    # df = psr.parse_to_dataframe('../training-data/sample_rules/valid_online.rules')
    df = psr.parse_to_dataframe('../training-data/dataset_rule_check/bad.rules')
except ImportError:
    print("Error: parse_suricata_rules not found. Please verify the path.")
    sys.exit(1)

print(f"Total rules loaded: {len(df)}")

# Run analysis
analysis_df = analyze_suricata_patterns(df)

# Unique patterns filter
unusual_rules = analysis_df[analysis_df['keyword_sequence'].map(analysis_df['keyword_sequence'].value_counts()) == 1]
print(f"\nFound {len(unusual_rules)} unique/rare rule structures.")

Analyze Length of Suricata Rules

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("parse_suricata_rules").resolve().parent.parent / "training-data/scripts"))

import parse_suricata_rules as psr
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

MIN_LENGTH_THRESHOLD = 200
TOP_N_FIELDS = 10
TOP_N_FIELD_VALUES = 10

# Load the rules DataFrame
df = psr.parse_to_dataframe('../training-data/dataset_rule_fix/good.rules')

# -----------------------------------------------------------------------------
# 1. Calculate the length of each field for each record
# -----------------------------------------------------------------------------
def calc_field_length(value):
    """Calculate the length of a field value (handle None/NaN)."""
    if pd.isna(value):
        return 0
    return len(str(value))

# Create a DataFrame with lengths for each field
df_lengths = df.apply(lambda col: col.apply(calc_field_length))

# Add a column for total length per record
df_lengths['total_length'] = df_lengths.sum(axis=1)

print(f"Total records: {len(df_lengths)}")
print(f"Records with total_length <= {MIN_LENGTH_THRESHOLD}: {(df_lengths['total_length'] <= MIN_LENGTH_THRESHOLD).sum()}")
print(f"Records with total_length > {MIN_LENGTH_THRESHOLD}: {(df_lengths['total_length'] > MIN_LENGTH_THRESHOLD).sum()}")

# -----------------------------------------------------------------------------
# 2. Filter out records with total_length < MIN_LENGTH_THRESHOLD
# -----------------------------------------------------------------------------
df_filtered = df_lengths[df_lengths['total_length'] > MIN_LENGTH_THRESHOLD].copy()
print(f"\nFiltered DataFrame shape: {df_filtered.shape}")

# -----------------------------------------------------------------------------
# 3. Get top 10 fields with longest length for each record
# -----------------------------------------------------------------------------
# Exclude total_length from the field columns
field_cols = [col for col in df_filtered.columns if col != 'total_length']

def get_top_n_fields(row, n=10):
    """Get the top N fields with longest length for a record."""
    field_lengths = row[field_cols].sort_values(ascending=False)
    return field_lengths.head(n)

# Store top 10 fields per record
top_fields_per_record = []
for idx, row in df_filtered.iterrows():
    top_fields = get_top_n_fields(row)
    top_fields_per_record.append({
        'record_idx': idx,
        'top_fields': list(top_fields.index),
        'top_lengths': list(top_fields.values)
    })

# Display sample of top fields per record
print("\n--- Top 10 fields per record (first 5 records) ---")
for i, rec in enumerate(top_fields_per_record[:5]):
    print(f"Record {rec['record_idx']}:")
    for field, length in zip(rec['top_fields'], rec['top_lengths']):
        print(f"  {field}: {length}")

# -----------------------------------------------------------------------------
# 4. Aggregate to determine top N fields (TOP_N_FIELDS) with longest length among all records
# -----------------------------------------------------------------------------
# Approach: Rank by maximum (longest individual value) length per field
field_maxes = df_filtered[field_cols].max().sort_values(ascending=False)
top_n_fields_overall = field_maxes.head(TOP_N_FIELDS)

print(f"\n--- Top {TOP_N_FIELDS} Fields by Max Length (longest individual value) ---")
print(top_n_fields_overall.to_string())

# Alternative: Count how often each field appears in top N per record
field_rank_counts = Counter()
for rec in top_fields_per_record:
    for field in rec['top_fields']:
        field_rank_counts[field] += 1

top_n_by_frequency = dict(field_rank_counts.most_common(TOP_N_FIELDS))
print(f"\n--- Top {TOP_N_FIELDS} Fields by Frequency in Top {TOP_N_FIELDS} per Record ---")
for field, count in top_n_by_frequency.items():
    print(f"  {field}: {count}")

top_n_field_names = list(top_n_fields_overall.index)

# -----------------------------------------------------------------------------
# 5. Create a graph to represent the distribution
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Top N fields by total length
ax1 = axes[0, 0]
top_n_fields_overall.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Max Length (longest individual value)')
ax1.set_ylabel('Field')
ax1.set_title(f'Top {TOP_N_FIELDS} Fields by Total Length')
ax1.invert_yaxis()

# Plot 2: Top N fields by frequency in top N per record
ax2 = axes[0, 1]
pd.Series(top_n_by_frequency).plot(kind='barh', ax=ax2, color='coral')
ax2.set_xlabel(f'Frequency (count in top {TOP_N_FIELDS} per record)')
ax2.set_ylabel('Field')
ax2.set_title(f'Top {TOP_N_FIELDS} Fields by Frequency in Top {TOP_N_FIELDS} per Record')
ax2.invert_yaxis()

# Plot 3: Distribution of total_length
ax3 = axes[1, 0]
df_lengths['total_length'].hist(bins=50, ax=ax3, color='green', edgecolor='black', alpha=0.7)
ax3.axvline(x=200, color='red', linestyle='--', label='Threshold (200)')
ax3.set_xlabel('Total Length')
ax3.set_ylabel('Number of Records')
ax3.set_title('Distribution of Total Length per Record')
ax3.legend()

# Plot 4: Box plot of field lengths for top N fields
ax4 = axes[1, 1]
top_n_field_names = list(top_n_fields_overall.index)
df_filtered[top_n_field_names].boxplot(ax=ax4, vert=False)
ax4.set_xlabel('Length')
ax4.set_ylabel('Field')
ax4.set_title(f'Length Distribution of Top {TOP_N_FIELDS} Fields')

plt.tight_layout()
plt.show()

print(f"\n--- Summary Table: Top {TOP_N_FIELDS} Fields ---")
summary_df = pd.DataFrame({
    'Field': top_n_fields_overall.index,
    'Max_Length': top_n_fields_overall.values,
    'Mean_Length': df_filtered[top_n_field_names].mean().values,
    'Total_Length': df_filtered[top_n_field_names].sum().values,
    f"Frequency_in_Top{TOP_N_FIELDS}": [field_rank_counts.get(f, 0) for f in top_n_field_names]
})
print(summary_df.to_string(index=False))

# -----------------------------------------------------------------------------
# 5. List the longest values for each top field
# -----------------------------------------------------------------------------
# Use the original df (not df_lengths) to get actual field values, filtered to same records
df_values = df.loc[df_filtered.index]

print(f"\n--- Top {TOP_N_FIELD_VALUES} Longest Values per Top Field ---")
separator = "-" * 80
for field in top_n_field_names:
    longest = (
        df_values[field]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .iloc[df_values[field].dropna().astype(str).drop_duplicates().apply(len).argsort()[::-1]]
        .head(TOP_N_FIELD_VALUES)
        .reset_index(drop=True)
    )
    print(f"\n{field}")
    print(separator)
    for val in longest:
        print(val)
    print(separator)


Check all Rule Options Present in Suricata Rules

In [ ]:
import re

rules_file = '../training-data/dataset_rule_fix/good.rules'

standalone_kws = set()
empty_str_kws = set()

with open(rules_file, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Extract the options section: everything between the outermost ( and )
        m = re.search(r'\((.+)\)\s*$', line)
        if not m:
            continue
        opts = m.group(1)

        # Tokenize: split on ";" but be careful about quoted strings
        # Simple approach: use regex to find all option tokens
        # Options are separated by ";" — split naively but handle quoted strings
        tokens = []
        current = ""
        in_quote = False
        quote_char = None
        for ch in opts:
            if in_quote:
                current += ch
                if ch == quote_char:
                    in_quote = False
            elif ch in ('"', "'"):
                in_quote = True
                quote_char = ch
                current += ch
            elif ch == ';':
                tokens.append(current.strip())
                current = ""
            else:
                current += ch
        if current.strip():
            tokens.append(current.strip())

        for token in tokens:
            token = token.strip()
            if not token:
                continue
            # Standalone keyword: no colon → keyword;
            if ':' not in token:
                if re.match(r'^[a-zA-Z_][a-zA-Z0-9_.]*$', token):
                    standalone_kws.add(token)
            else:
                # keyword:"" → empty double-quoted string value
                kw_match = re.match(r'^([a-zA-Z_][a-zA-Z0-9_.]*):""$', token)
                if kw_match:
                    empty_str_kws.add(kw_match.group(1))

print("=== Standalone keywords (keyword;) ===")
for kw in sorted(standalone_kws):
    print(f"  {kw}")

print(f"\n=== Empty string keywords (keyword:\"\"\") ===")
for kw in sorted(empty_str_kws):
    print(f"  {kw}")

all_kws = sorted(standalone_kws | empty_str_kws)
print(f"\nTotal unique keywords: {len(all_kws)}")
print("\nComma-separated:")
print(",".join(all_kws))